# VIR Vbeta - G2 Throughput (Burn CUDA vs PyTorch)
Ukur tok/s forward VQ di GPU Kaggle.

In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']

In [ ]:
import subprocess, json, time, os
t0=time.time()
subprocess.run(['git','clone','--depth','1','https://github.com/Gomobra/VIR.git','/tmp/VIR'], check=True)
env={**os.environ,'CARGO_TARGET_DIR':'/tmp/target'}
b=subprocess.run(['cargo','build','--release','--manifest-path','/tmp/VIR/crates/vir-vq/Cargo.toml','--features','cuda','--bin','bench'], capture_output=True, text=True, env=env)
print('build rc', b.returncode)
print(b.stderr[-1500:])
r=subprocess.run(['/tmp/target/release/bench'], capture_output=True, text=True, env=env)
print('BURN:', r.stdout.strip())
print(r.stderr[-500:])
open('/kaggle/working/result.json','w').write(json.dumps({'burn': r.stdout.strip(), 'elapsed': time.time()-t0}))

In [ ]:
# PyTorch baseline (forward only)
import torch, time
dev='cuda'
V,D=4096,384
E=torch.randn(V,D,device=dev)
z=torch.randn(256*128,D,device=dev)
for _ in range(5):
    d=torch.cdist(z,E)
    idx=d.argmin(1)
torch.cuda.synchronize()
t0=time.time()
iters=50
for _ in range(iters):
    d=torch.cdist(z,E)
    idx=d.argmin(1)
torch.cuda.synchronize()
el=time.time()-t0
print(f'PYTORCH: vq_forward toks/s = {256*128*iters/el:.2f} (elapsed {el:.3f}s)')